# 05.1 Capstone: Tabular Classification / 综合项目：表格分类

这一份 notebook 是一个更接近真实项目节奏的综合练习。  
This notebook is a capstone exercise that is closer to the rhythm of a real project.

目标不是只训练一个模型，而是完整走一遍：  
The goal is not just to train one model, but to walk through the full loop:

- 问题定义 / problem definition
- 数据处理 / data processing
- baseline / 基线模型
- improved model / 改进模型
- 对照实验 / controlled experiments
- 结果分析 / result analysis
- 最终总结 / final summary

## 学习目标 / Learning Goals

学完后你应该能 / After this notebook, you should be able to:

1. 把一个表格分类任务完整跑通 / Run a tabular classification task end to end.
2. 对比 `baseline / 基线模型` 和 `MLP / 多层感知机` / Compare a baseline model with an `MLP`.
3. 组织训练、验证、测试三部分流程 / Organize train, validation, and test workflows.
4. 整理对照实验结果表 / Build a comparison table for controlled experiments.
5. 用 `confusion matrix / 混淆矩阵` 和误分类样本分析结果 / Analyze results with a confusion matrix and misclassified samples.
6. 写出一份像项目总结一样的结论 / Write a conclusion that looks like a real project summary.

In [ ]:
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(42)

## 1. 问题定义 / Problem Definition

这里使用 `breast cancer` 数据集做二分类任务。  
We use the `breast cancer` dataset here for a binary classification task.

任务目标 / Task goal:

- 根据表格特征预测肿瘤类别 / predict the tumor class from tabular features

这个数据集的优点是 / The advantages of this dataset are:

- 规模小，适合教学 / small enough for teaching
- 是标准二分类 / a standard binary classification task
- 特征都是数值型，适合 `MLP` / all features are numeric, so it fits an `MLP` well

In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df.rename(columns={"target": "label"}, inplace=True)

print("dataset shape =", df.shape)
print("label counts =\n", df["label"].value_counts())
print("target names =", list(data.target_names))
df.head(3)

## 2. 数据拆分与标准化 / Data Split and Standardization

表格数据里一个很常见的预处理步骤是 `standardization / 标准化`。  
A very common preprocessing step for tabular data is `standardization`.

这里我们严格按这个顺序处理 / We follow this order strictly:

1. 先拆分训练 / 验证 / 测试 / split into train / validation / test first
2. 只在训练集上 `fit scaler` / fit the scaler only on the training split
3. 再把同一个 scaler 应用到验证和测试集 / apply the same scaler to validation and test

In [ ]:
X = df.drop(columns=["label"])
y = df["label"]
feature_names = list(X.columns)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full,
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("train shape =", X_train_scaled.shape)
print("val shape =", X_val_scaled.shape)
print("test shape =", X_test_scaled.shape)
print("train positive rate / 训练集正样本比例 =", y_train.mean())

## 3. Baseline: Logistic Regression / 基线模型：逻辑回归

`Logistic Regression / 逻辑回归` 是表格分类里非常常见的 baseline。  
`Logistic Regression` is a very common baseline for tabular classification.

它的作用不是“最先进”，而是提供一个清晰、可靠、好解释的参照系。  
Its role is not to be state of the art, but to provide a clear, reliable, and interpretable reference point.

In [ ]:
baseline_model = LogisticRegression(max_iter=3000, random_state=42)
baseline_model.fit(X_train_scaled, y_train)

baseline_val_preds = baseline_model.predict(X_val_scaled)
baseline_test_preds = baseline_model.predict(X_test_scaled)

baseline_val_acc = accuracy_score(y_val, baseline_val_preds)
baseline_test_acc = accuracy_score(y_test, baseline_test_preds)

print("baseline val acc =", round(baseline_val_acc, 4))
print("baseline test acc =", round(baseline_test_acc, 4))

## 4. 用 PyTorch 构建数据管线 / Build the PyTorch Data Pipeline

接下来进入 `PyTorch` 版本的模型实验。  
Next we move to the `PyTorch` experiments.

这里依然沿用已经标准化好的数值特征。  
We continue using the already standardized numeric features.

In [ ]:
train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train.values, dtype=torch.long),
)
val_ds = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(y_val.values, dtype=torch.long),
)
test_ds = TensorDataset(
    torch.tensor(X_test_scaled, dtype=torch.float32),
    torch.tensor(y_test.values, dtype=torch.long),
)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

xb, yb = next(iter(train_loader))
print("xb.shape =", xb.shape)
print("yb.shape =", yb.shape)

## 5. 模型、训练函数与评估函数
## Model, Training Function, and Evaluation Function

为了公平比较，我们让不同 `MLP` 共享同一套训练逻辑。  
To keep the comparison fair, we let different `MLP` variants share the same training logic.

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim=32, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for xb, yb in loader:
            logits = model(xb)
            loss = loss_fn(logits, yb)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_loss += loss.item() * xb.size(0)
            total_correct += (preds == yb).sum().item()
            total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


def train_torch_model(config):
    set_seed(42)
    model = TabularMLP(
        in_dim=X_train_scaled.shape[1],
        hidden_dim=config["hidden_dim"],
        dropout=config["dropout"],
    )
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    history = []
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = -1.0

    for epoch in range(1, config["epochs"] + 1):
        train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
            }
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def collect_predictions(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)
            all_preds.append(preds)
            all_targets.append(yb)
    return torch.cat(all_preds), torch.cat(all_targets)

## 6. 对照实验 / Controlled Experiments

这里做两组 `MLP` 对照实验 / We run two `MLP` controlled experiments here:

- `MLP-Small`：较小隐藏层 / smaller hidden layer
- `MLP-Regularized`：更大隐藏层 + dropout + weight decay / larger hidden layer + dropout + weight decay

这样可以比较：模型容量和正则化是否带来收益。  
This lets us compare whether capacity and regularization bring gains.

In [ ]:
experiment_configs = {
    "MLP-Small": {
        "hidden_dim": 16,
        "dropout": 0.0,
        "lr": 0.01,
        "weight_decay": 0.0,
        "epochs": 25,
    },
    "MLP-Regularized": {
        "hidden_dim": 64,
        "dropout": 0.2,
        "lr": 0.01,
        "weight_decay": 1e-4,
        "epochs": 30,
    },
}

torch_runs = {}
for name, cfg in experiment_configs.items():
    model, history_df = train_torch_model(cfg)
    test_preds, test_targets = collect_predictions(model, test_loader)
    val_preds, val_targets = collect_predictions(model, val_loader)
    torch_runs[name] = {
        "config": cfg,
        "model": model,
        "history": history_df,
        "val_acc": accuracy_score(val_targets.numpy(), val_preds.numpy()),
        "test_acc": accuracy_score(test_targets.numpy(), test_preds.numpy()),
        "test_preds": test_preds.numpy(),
        "test_targets": test_targets.numpy(),
    }
    print(name, "best val acc =", round(torch_runs[name]["val_acc"], 4), "| test acc =", round(torch_runs[name]["test_acc"], 4))

## 7. 结果表 / Result Table

一个项目最少也应该有一张清晰的对比表。  
A project should at minimum have a clear comparison table.

In [ ]:
results_df = pd.DataFrame(
    [
        {
            "model": "LogisticRegression",
            "category": "baseline",
            "val_acc": round(baseline_val_acc, 4),
            "test_acc": round(baseline_test_acc, 4),
            "notes": "classic tabular baseline",
        },
        {
            "model": "MLP-Small",
            "category": "torch",
            "val_acc": round(torch_runs["MLP-Small"]["val_acc"], 4),
            "test_acc": round(torch_runs["MLP-Small"]["test_acc"], 4),
            "notes": "smaller hidden layer",
        },
        {
            "model": "MLP-Regularized",
            "category": "torch",
            "val_acc": round(torch_runs["MLP-Regularized"]["val_acc"], 4),
            "test_acc": round(torch_runs["MLP-Regularized"]["test_acc"], 4),
            "notes": "larger hidden layer + dropout + weight decay",
        },
    ]
).sort_values(by=["test_acc", "val_acc"], ascending=False)
results_df

## 8. 训练曲线 / Training Curves

这里只画 `PyTorch` 模型的曲线，因为 baseline 没有 epoch-by-epoch history。  
We only plot curves for the `PyTorch` models here because the baseline does not have epoch-by-epoch history.

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
for name, run in torch_runs.items():
    hist = run["history"]
    plt.plot(hist["epoch"], hist["train_loss"], label=f"{name} train")
    plt.plot(hist["epoch"], hist["val_loss"], linestyle="--", label=f"{name} val")
plt.title("Loss Curves")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
for name, run in torch_runs.items():
    hist = run["history"]
    plt.plot(hist["epoch"], hist["train_acc"], label=f"{name} train")
    plt.plot(hist["epoch"], hist["val_acc"], linestyle="--", label=f"{name} val")
plt.title("Accuracy Curves")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()
plt.close()

## 9. 选择最优模型 / Select the Best Model

这里按 `test_acc` 最高来选最终模型，仅用于教学展示。  
Here we select the final model by the highest `test_acc`, only for teaching demonstration.

真实项目里通常应优先按验证集选型，再只用测试集做一次最终报告。  
In real projects, you usually choose by validation performance and use the test set only once for final reporting.

In [ ]:
all_candidates = {
    "LogisticRegression": {
        "test_preds": baseline_test_preds,
        "test_targets": y_test.to_numpy(),
        "test_acc": baseline_test_acc,
    },
    "MLP-Small": {
        "test_preds": torch_runs["MLP-Small"]["test_preds"],
        "test_targets": torch_runs["MLP-Small"]["test_targets"],
        "test_acc": torch_runs["MLP-Small"]["test_acc"],
    },
    "MLP-Regularized": {
        "test_preds": torch_runs["MLP-Regularized"]["test_preds"],
        "test_targets": torch_runs["MLP-Regularized"]["test_targets"],
        "test_acc": torch_runs["MLP-Regularized"]["test_acc"],
    },
}

best_name = max(all_candidates, key=lambda name: all_candidates[name]["test_acc"])
best_preds = all_candidates[best_name]["test_preds"]
best_targets = all_candidates[best_name]["test_targets"]

print("best model =", best_name)
print("best test acc =", round(all_candidates[best_name]["test_acc"], 4))

## 10. Confusion Matrix / 混淆矩阵

二分类里，`confusion matrix` 能帮助你看到：  
In binary classification, a `confusion matrix` helps you see:

- 哪类样本更容易被错分 / which class is easier to misclassify
- 错误主要是 `false positive` 还是 `false negative` / whether the main mistakes are `false positives` or `false negatives`

In [ ]:
cm = confusion_matrix(best_targets, best_preds)
plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title(f"Confusion Matrix: {best_name}")
plt.xlabel("predicted label")
plt.ylabel("true label")
plt.colorbar()

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()
plt.close()

print(classification_report(best_targets, best_preds, digits=4, target_names=list(data.target_names)))

## 11. 误分类样本分析 / Misclassified Sample Analysis

项目里很重要的一步是：不要只停在整体指标。  
An important project step is: do not stop at overall metrics.

下面看几条误分类样本，以及其中部分关键特征。  
Below we inspect several misclassified samples and a few key features.

In [ ]:
test_index = y_test.index.to_numpy()
mis_positions = np.where(best_preds != best_targets)[0]
selected_positions = mis_positions[:8]
selected_columns = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
]

if len(selected_positions) > 0:
    mis_df = X_test.iloc[selected_positions][selected_columns].copy()
    mis_df["true_label"] = y_test.iloc[selected_positions].to_numpy()
    mis_df["pred_label"] = best_preds[selected_positions]
    mis_df
else:
    print("No misclassified samples / 没有误分类样本。")

## 12. 最终结论 / Final Conclusion

一个够实用的项目结论，至少要回答 / A practical project conclusion should at least answer:

1. 最终哪个模型最好 / which model is best in the end
2. 相比 baseline 有没有提升 / whether there was improvement over the baseline
3. 主要错误在哪里 / where the main errors are
4. 下一步应该做什么 / what the next steps should be

In [ ]:
summary_lines = [
    f"Best model: {best_name}",
    f"Baseline test accuracy: {baseline_test_acc:.4f}",
    f"Best test accuracy: {all_candidates[best_name]['test_acc']:.4f}",
    f"Improvement over baseline: {all_candidates[best_name]['test_acc'] - baseline_test_acc:.4f}",
    f"Number of misclassified test samples: {int((best_preds != best_targets).sum())}",
]

for line in summary_lines:
    print(line)

In [ ]:
# 练习 1 / Exercise 1
# 如果 MLP 的 train_acc 很高，但 val_acc 比 baseline 还低，
# If the MLP has very high train_acc but lower val_acc than the baseline,
# 你会先怀疑什么？
# what would you suspect first?

练习 1 参考答案 / Exercise 1 Reference Answer

首先会怀疑 `overfitting / 过拟合`。  
I would first suspect `overfitting`.

然后可以考虑更强正则化、更小模型、或更好的特征处理。  
Then I would consider stronger regularization, a smaller model, or better feature processing.

In [ ]:
# 练习 2 / Exercise 2
# 为什么在表格任务里，LogisticRegression 常常值得先跑？
# Why is LogisticRegression often worth running first on tabular tasks?

练习 2 参考答案 / Exercise 2 Reference Answer

因为它简单、稳定、速度快，而且常常已经能提供很强的 baseline。  
Because it is simple, stable, fast, and often already provides a strong baseline.

如果连 baseline 都打不过，通常说明后面的复杂模型还没有真正带来价值。  
If you cannot beat the baseline, it often means the more complex model has not yet added real value.

## 13. 小结 / Summary

这一份综合项目最重要的不是某个具体分数，而是你完成了完整项目流程。  
The most important outcome of this capstone is not a specific score, but the fact that you completed the full project workflow.

你已经练习了 / You have practiced:

1. 问题定义 / problem definition
2. 数据拆分与标准化 / data splitting and standardization
3. baseline 与改进模型对比 / comparing a baseline and improved models
4. 结果表、训练曲线、混淆矩阵 / result tables, training curves, and confusion matrices
5. 误分类分析与结论书写 / misclassification analysis and conclusion writing